In [1]:
import os
import sys
import random
import pandas as pd
from datetime import datetime
import openai
import mlflow
from tqdm.auto import tqdm

mlflow_tracking_uri = os.environ.get('MLFLOW_TRACKING_URI', 'https://mlflow.sampadhegde.in')
mlflow_experiment = os.environ.get('MLFLOW_EXPERIMENT_NAME', 'torrent-naming')
try:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment)
    mlflow.openai.autolog()
    print(f'MLflow LLM tracing enabled: {mlflow_tracking_uri} ({mlflow_experiment})')
except Exception as error:
    print(f'MLflow tracing could not be initialized: {error}')

tqdm.pandas()

MLflow LLM tracing enabled: https://mlflow.sampadhegde.in (torrent-naming)


In [ ]:
base_path = '/Volumes/Crucial X8/Torrent Backup/'
model = 'google/gemma-4-12b'
base_url = 'http://192.168.0.119:1234/v1'

In [ ]:
client = openai.OpenAI(base_url=base_url, api_key='NA')

In [ ]:
client.models.list()

In [ ]:
files = os.listdir(base_path)
files[:2]

In [ ]:
def detect_file_folder(path: str):
    if os.path.isfile(path):
        return 'file'
    elif os.path.isdir(path):
        return 'folder'
    else:
        return 'unknown'
    
def get_extension(filename: str):
    return filename.split('.')[-1] if '.' in filename else ''

In [ ]:
def clean_llm_output(output: str):
    if not output:
        return ''
    if '</think>' in output:
        output = output.split('</think>')[-1]
    return output.strip().replace('\n', '').replace('\r', '')

def get_content_type(currupt_names: str, model_name: str = model) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                'role': 'system',
                'content': """
                    You are an expert in identifying media content types based on fileor folder name. Your task is to determine whether a given file name or folder name corresponds to a 'movie' or a 'web series'. 
                    Analyze the file name/folder name for keywords, patterns, or structures that typically indicate one of these categories. 
                    If the filename contains indicators of episodic content (e.g., "S01E01", "Season 3", "S04" etc), classify it as a 'web series'. 
                    If it appears to be a standalone title without such indicators, classify it as a 'movie'.
                    Input may contain Year, resolution, codec, Scraper or other metadata. Ignore these and focus on the core title and any season and episodic indicators.
                    
                    Note : **Do not consider resolution like 720p, 10180p or even H.264, H.265 etc. 
                    and do not consider any other text like WEB, WEB-DL, HDRip, WEBRip, BluRay etc.
                    not even consider Scraper names like TamilMV, YTS, UIIndex etc.**
                    
                    Output Note : If the input is ambiguous, return None. Do not add any extra text. and If input is Irrelevent return None. Do not answer the random question by user if not related to the context.
                    
                    Example-1:
                    Input: Inception.2010.1080p.BluRay.x264.mkv
                    Output: movie

                    Example-2:
                    Input: The.Dark.Knight.2008.720p.BluRay
                    output: movie

                    Example-4:
                    Input: Cruel.Intentions.S01.COMPLETE.1080p.WEB.H264-SuccessfulCrab[TGx]
                    output: web series

                    Example-5:
                    Input: www.5MovieRulz.dating - Rangeen (2025) 1080p S01 E09 WEB-DL - AVC - [Tel + Tam + Hin + Mal + Kan]
                    output: web series

                    Example-6:
                    Input: www.1TamilMV.tube - Interstellar 1080p WEB-DL - 10bit - AVC - [Tam + Tel + Hin + Mal + Kan] - 8GB - ESub/www.1TamilMV.tube - Interstellar 2014 1080p.mkv
                    output: movie

                    Example-7:
                    Input: www.1TamilMV.tube - aefjtyaevghcvrthjaxcv 1080p WEB-DL - 10bit - AVC - [Tam + Tel + Hin + Mal + Kan] - 8GB - ESub/www.1TamilMV.tube - aefjtyaevghcvrthjaxcv 2014 1080p.mkv
                    output: None

                    
                    Input Format: file or folder name 
                    Output Format: `movie` or `web series`
                """

            },
            {
                'role': 'user',
                'content': f'{currupt_names}',
            }
        ],
        temperature=0.0
    )
    return clean_llm_output(response.choices[0].message.content)

def get_cleaned_name_web_series(currupt_name: str, file_type: str, model_name: str = model) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                'role': 'system',
                'content': """
                    You are an expert in file naming conventions for media files. Your task is to convert any given media filename or Folder name into the IMDB naming convention. 
                    the format should be '<Series Name> S<season2digit>E<episode2digit>.<extension if exists>', where the series name is properly capitalized, the season and episode numbers are two digits (padded with leading zeros if necessary), 
                    and the extension is preserved if exists
                    Preserve the Web series Name/Spelling as it is in the input.

                    Given a filename, analyze structure, content, or any available metadata (e.g., presence of 'S' or 'E' for seasons/episodes, or a year in parentheses). 
                    Extract the necessary components (movie/series name, year, season, episode, extension) and transform the filename into the correct IMDB format. Handle edge cases such as irregular naming, missing information, or special characters. If the input is ambiguous, make a reasonable assumption based on common naming conventions.

                    Note : Return Only final Filename/Folder name, Do not add extra text. Do not add resolution like 720p, 10180p or even H.264, 265 etc. and do not add any other text like WEB, BluRay etc not even the Scraper name. Discard Year. date or any other information if present in the input which is Irrelevent.
                    
                    Example-1:
                    Input: file $$ The Flash 2013 Season4 Episode3 h265.avi
                    Output: The Flash S04E03.avi.

                    Example-2:
                    Input: folder $$ Cruel.Intentions.S01.COMPLETE.1080p.WEB.H264-SuccessfulCrab[TGx]
                    output: Cruel Intentions S01

                    Example-3:
                    Input: folder $$ www.5MovieRulz.dating - Rangeen (2025) 1080p S01 EP(01-09) WEB-DL - AVC - [Tel + Tam + Hin + Mal + Kan]
                    output: Rangeen S01

                    Example-4:
                    Input: file $$ www.1TamilMV.tube - Special OPS (2025) S02 EP(01-07) WEB-DL - 1080p - AVC - [Tam + Tel + Hin + Mal + Kan] - 9GB - ESub/www.1TamilMV.tube - Special Ops S02E07 Thursday 1080p.mkv
                    output: Special Ops S02E07.mkv

                    Input Format: <`File` or `Folder`> $$ filename
                    Output Format: IMDB Formatted Name
                """

            },
            {
                'role': 'user',
                'content': f'{file_type} $$ {currupt_name}',
            }
        ],
        temperature=0.0
    )
    return clean_llm_output(response.choices[0].message.content)

def get_cleaned_name_movie(currupt_name: str, file_type: str, model_name: str = model) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                'role': 'system',
                'content': """
                    You are an expert in file naming conventions for media files. Your task is to convert any given media filename into the IMDB naming convention. 
                    For movies, the format should be '<Movie Name> (year).<extension>', where the movie name is properly capitalized, the year is a four-digit number in parentheses, and the extension is preserved if Exists.
                    Preserve the Web series Name/Spelling as it is in the input.

                    Given a filename or Folder Name, analyze structure, content, or any available metadata. 
                    Handle edge cases such as irregular naming, missing information, or special characters. If the input is ambiguous, make a reasonable assumption based on common naming conventions.
                    
                    Note : Return Only final Filename/Folder name, Do not add extra text. Do not add resolution like 720p, 10180p or even H.264, 265 etc. and do not add any other text like WEB, BluRay etc not even the Scraper name.
                    
                    Example-1:
                    Input: file $$ Inception.2010.1080p.BluRay.x264.mkv
                    Output: Inception (2010).mkv

                    Example-2:
                    Input: folder $$ The.Dark.Knight.2008.720p.BluRay
                    output: The Dark Knight (2008)

                    Example-3:
                    Input: folder $$ www.5MovieRulz.dating - Avatar The Way of Water (2022) 1080p WEB-DL - AVC - [Tel + Tam + Hin + Mal + Kan]
                    output: Avatar The Way of Water (2022)

                    Example-4:
                    Input: file $$ www.1TamilMV.tube - Interstellar 1080p WEB-DL - 10bit - AVC - [Tam + Tel + Hin + Mal + Kan] - 8GB - ESub/www.1TamilMV.tube - Interstellar 2014 1080p.mkv
                    output: Interstellar (2014).mkv

                    Example-5:
                    Input: file $$ The.Matrix.1999.REMASTERED.1080p.BluRay.x264.mkv
                    output: The Matrix (1999).mkv

                    Example-6:
                    Input: file $$ www.5MovieRulz.gd - Kannur Squad (2023) 1080p WEB-DL - AVC - (DD+5.1 - 192Kbps) [Tel + Hin + Kan] - 2.8GB - ESub.mkv
                    output: Kannur Squad (2023).mkv

                    
                    Input Format: <`File` or `Folder`> $$ filename
                    Output Format: IMDB Formatted Name
                """

            },
            {
                'role': 'user',
                'content': f'{file_type} $$ {currupt_name}',
            }
        ],
        temperature=0.0
    )
    return clean_llm_output(response.choices[0].message.content)

In [ ]:
def rename_movies(base_path, path, clean_name):
    if os.path.exists(os.path.join(base_path, clean_name)):
        print(f"Skipped: {clean_name} already exists.")
        return
    if os.path.isfile(os.path.join(base_path, path)):
        try:
            os.rename(os.path.join(base_path, path), os.path.join(base_path, clean_name))
        except Exception as e:
            print(f"Error renaming file {path} to {clean_name}: {e}")
    elif os.path.isdir(os.path.join(base_path, path)):
        try:
            os.rename(os.path.join(base_path, path), os.path.join(base_path, clean_name))
        except Exception as e:
            print(f"Error renaming folder {path} to {clean_name}: {e}")
        for inner_file in os.listdir(os.path.join(base_path, clean_name)):
            if inner_file.split('.')[-1] not in ['mkv', 'mp4', 'avi', 'mov', 'wmv', 'flv', 'mkv', 'mp3', 'wav', 'flac', 'aac', 'srt', 'sub']:
                try:
                    # os.remove(os.path.join(base_path, clean_name, inner_file))
                    print(f"Removed file {inner_file} in folder {clean_name}")
                except Exception as e:
                    print(f"Error removing file {inner_file} in folder {clean_name}: {e}")
            else:
                try:
                    os.rename(os.path.join(base_path, clean_name, inner_file), os.path.join(base_path, clean_name, clean_name + '.' + inner_file.split('.')[-1]))
                    print(f"Renamed inner file {inner_file} to {clean_name + '.' + inner_file.split('.')[-1]} in folder {clean_name}")
                except Exception as e:
                    print(f"Error renaming inner file {inner_file} in folder {clean_name}: {e}")

def rename_web_series(base_path, path, clean_name):
    if os.path.exists(os.path.join(base_path, clean_name)):
        print(f"Skipped: {clean_name} already exists.")
        return
    if os.path.isfile(os.path.join(base_path, path)):
        try:
            os.rename(os.path.join(base_path, path), os.path.join(base_path, clean_name))
        except Exception as e:
            print(f"Error renaming file {path} to {clean_name}: {e}")
    elif os.path.isdir(os.path.join(base_path, path)):
        try:
            os.rename(os.path.join(base_path, path), os.path.join(base_path, clean_name))
        except Exception as e:
            print(f"Error renaming folder {path} to {clean_name}: {e}")
        for inner_file in os.listdir(os.path.join(base_path, clean_name)):
            if inner_file.split('.')[-1] not in ['mkv', 'mp4', 'avi', 'mov', 'wmv', 'flv', 'mkv', 'mp3', 'wav', 'flac', 'aac', 'srt', 'sub']:
                try:
                    # os.remove(os.path.join(base_path, clean_name, inner_file))
                    pass
                except Exception as e:
                    print(f"Error removing file {inner_file} in folder {clean_name}: {e}")
            else:
                episode_name = get_cleaned_name_web_series(inner_file, 'file')
                try:
                    os.rename(os.path.join(base_path, clean_name, inner_file), os.path.join(base_path, clean_name, episode_name))
                    print(f"Renamed inner episode {inner_file} to {episode_name} in folder {clean_name}")
                except Exception as e:
                    print(f"Error renaming inner file {inner_file} in folder {clean_name}: {e}")

def create_folder_movies(base_path):
    for file in os.listdir(base_path):
        if os.path.isfile(os.path.join(base_path, file)):
            folder_name = '.'.join(file.split('.')[:-1])
            try:
                print('Fodler name : ', folder_name, ' For file : ', file)
                os.mkdir(os.path.join(base_path, folder_name))
                os.rename(os.path.join(base_path, file), os.path.join(base_path, folder_name, file))
                print(f'Created folder {folder_name} and moved file {file} into it.')
            except Exception as e:
                print(f"** Error creating folder {folder_name} or moving file {file}: {e}")

def create_folder_S01(base_path):
    for file in os.listdir(base_path):
        if os.path.isdir(os.path.join(base_path, file)) and 'S01' in file:
            folder_name = file.replace(' S01', '')
            print('Fodler name : ', folder_name, 'For file : ', file)
            os.mkdir(os.path.join(base_path, folder_name))
            os.rename(os.path.join(base_path, file), os.path.join(base_path, folder_name, file))
            print(f'Created folder {folder_name} and moved file {file} into it.')

def apply_renaming(df: pd.DataFrame, base_path: str):
    total = len(df)
    for i, (index, row) in enumerate(tqdm(df.iterrows(), total=total, desc="Applying renaming"), 1):
        if row['content_type'] == 'movie':
            rename_movies(base_path, row['Original Name'], row['cleaned_name'])
        elif row['content_type'] == 'web series':
            rename_web_series(base_path, row['Original Name'], row['cleaned_name'])
        else:
            print(f"[{i}/{total}] Skipped renaming for {row['Original Name']} due to unknown content type.")

In [ ]:
df = pd.DataFrame(files, columns=['Original Name'])
df['Type'] = df['Original Name'].apply(lambda x: detect_file_folder(os.path.join(base_path, x)))
df['extension'] = df['Original Name'].apply(get_extension)
df.head()

In [ ]:
# Test code
idx = random.randint(0, len(df)-1)
content_type = get_content_type(df.iloc[idx]['Original Name'])
if 'movie' in content_type.lower():
    cleaned_name = get_cleaned_name_movie(df.iloc[idx]['Original Name'], df.iloc[idx]['Type'])
elif 'web series' in content_type.lower():
    cleaned_name = get_cleaned_name_web_series(df.iloc[idx]['Original Name'], df.iloc[idx]['Type'])
else:
    cleaned_name = 'None'
print(f'Content Type: {content_type},\nCleaned Name: {cleaned_name},\nOriginal Name: {df.iloc[idx]["Original Name"]}')

In [ ]:
total = len(df)
print(f"Detecting content type for {total} files/folders...")
content_types = []

for i, name in enumerate(tqdm(df['Original Name'], desc="Detecting content type"), 1):
    c_type = get_content_type(name)
    content_types.append(c_type)
    tqdm.write(f"[{i}/{total}] {name} -> {c_type}")

df['content_type'] = content_types
df.head()

In [ ]:
total = len(df)
print(f"Generating cleaned names for {total} items...")
cleaned_names = []

for i, (idx, row) in enumerate(tqdm(df.iterrows(), total=total, desc="Cleaning names"), 1):
    if row['content_type'] == 'movie':
        cleaned = get_cleaned_name_movie(row['Original Name'], row['Type'])
    elif row['content_type'] == 'web series':
        cleaned = get_cleaned_name_web_series(row['Original Name'], row['Type'])
    else:
        cleaned = None
    cleaned_names.append(cleaned)
    tqdm.write(f"[{i}/{total}] ({row['content_type']}) {row['Original Name']} -> {cleaned}")

df['cleaned_name'] = cleaned_names
df.head()

In [ ]:
df[df['cleaned_name'].str.len() > 40]

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import urllib.parse
from functools import lru_cache

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

def parse_media_title(raw_name):
    s = re.sub(r'\.(mkv|mp4|avi|mov|wmv|flv|srt|sub)$', '', str(raw_name), flags=re.IGNORECASE).strip()
    year_match = re.search(r'\b(19\d\d|20\d\d)\b', s)
    year = int(year_match.group(1)) if year_match else None
    
    season_match = re.search(r'\b(?:S|Season\s*)(\d+)\b', s, flags=re.IGNORECASE)
    season = int(season_match.group(1)) if season_match else None
    
    base = re.sub(r'\bS\d+(?:E\d+)?\b', '', s, flags=re.IGNORECASE)
    base = re.sub(r'\bSeason\s*\d+\b', '', base, flags=re.IGNORECASE)
    base = re.sub(r'\(\d{4}\)', '', base)
    base = re.sub(r'\[.*?\]', '', base)
    base = re.sub(r'\(.*?\)', '', base)
    base = re.sub(r'\s+', ' ', base).strip()
    
    return {
        'clean_name': s,
        'base_title': base,
        'year': year,
        'season': season
    }

def search_wikipedia_direct(base_title, year=None, season=None):
    queries = []
    if season:
        queries.append(f'{base_title}')
        queries.append(f'{base_title} season {season}')
    elif year:
        queries.append(f'{base_title} {year} film')
        queries.append(f'{base_title} {year}')
        queries.append(f'{base_title}')
    else:
        queries.append(f'{base_title}')

    url = 'https://en.wikipedia.org/w/api.php'
    candidates = []
    base_lower = base_title.lower()
    for q in queries:
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': q,
            'format': 'json',
            'srlimit': 5
        }
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=5)
            data = resp.json().get('query', {}).get('search', [])
            for item in data:
                title = item['title']
                t_lower = title.lower()
                if base_lower in t_lower or all(w in t_lower for w in base_lower.split() if len(w) > 2):
                    if title not in candidates:
                        candidates.append(title)
        except Exception:
            pass
        if len(candidates) >= 3:
            break
    return candidates

def extract_wikipedia_metadata(wiki_title, target_year=None):
    wiki_url = f'https://en.wikipedia.org/wiki/{urllib.parse.quote(wiki_title.replace(" ", "_"))}'
    try:
        resp = requests.get(wiki_url, headers=headers, timeout=5)
        soup = BeautifulSoup(resp.text, 'html.parser')
    except Exception:
        return {}

    infobox = soup.find('table', class_=lambda c: c and 'infobox' in c)
    if not infobox:
        return {}

    data = {}
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        key = th.text.strip().lower()
        val = td.get_text(' ', strip=True)
        if any(k in key for k in ['directed', 'director', 'creator', 'created by']):
            data.setdefault('director', val)
        elif 'starring' in key:
            data['cast'] = [x.strip() for x in val.split(',') if x.strip()]
        elif 'language' in key:
            data['language'] = val
            data['languages'] = [x.strip() for x in val.split(',') if x.strip()]
        elif any(k in key for k in ['release date', 'original release', 'aired']):
            data['release_date'] = val
            ym = re.search(r'\b(19\d\d|20\d\d)\b', val)
            if ym:
                data['year'] = int(ym.group(1))
        elif 'country' in key:
            data['country'] = val

    if target_year and 'year' in data:
        if abs(data['year'] - target_year) > 1:
            return {}

    title_tag = soup.find('h1', id='firstHeading')
    if title_tag:
        data['title'] = title_tag.text.strip()
    data['wiki_url'] = wiki_url
    return data

def search_imdb_direct(base_title, year=None):
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', base_title).strip('_').lower()
    if not slug:
        return None
    url = f'https://v3.sg.media-imdb.com/suggestion/{slug[0]}/{slug}.json'
    try:
        resp = requests.get(url, headers=headers, timeout=5)
        if resp.status_code == 200:
            items = [d for d in resp.json().get('d', []) if d.get('id', '').startswith('tt')]
            if not items:
                return None
            best = items[0]
            if year:
                for item in items:
                    if item.get('y') and abs(item.get('y') - year) <= 1:
                        best = item
                        break
            return {
                'title': best.get('l'),
                'year': str(best.get('y', '')),
                'cast': [c.strip() for c in best.get('s', '').split(',') if c.strip()],
                'type': best.get('qid'),
                'imdb_id': best.get('id'),
                'imdb_url': f'https://www.imdb.com/title/{best.get("id")}/',
            }
    except Exception:
        pass
    return None

def infer_language_with_llm(title, year='', cast=''):
    try:
        prompt = f'What is the primary original spoken language of the film or TV show \"{title}\"'
        if year:
            prompt += f' ({year})'
        if cast:
            prompt += f' starring {cast}'
        prompt += '? Reply ONLY with the language name (e.g. English, Hindi, Malayalam, Tamil, Tagalog, Spanish, Korean, Japanese, French). Do not add any punctuation or extra words.'
        res = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.0
        )
        ans = res.choices[0].message.content.strip()
        ans = re.sub(r'[^a-zA-Z\s]', '', ans).strip()
        return ans
    except Exception:
        return ''

@lru_cache(maxsize=2048)
def get_movie_metadata(movie_name):
    parsed = parse_media_title(movie_name)
    base_title = parsed['base_title']
    year = parsed['year']
    season = parsed['season']

    # 1. Search Wikipedia Direct
    wiki_candidates = search_wikipedia_direct(base_title, year=year, season=season)
    for cand in wiki_candidates:
        meta = extract_wikipedia_metadata(cand, target_year=year)
        if meta and ('director' in meta or 'language' in meta or 'cast' in meta):
            return meta

    # 2. Search IMDb Direct
    imdb_meta = search_imdb_direct(base_title, year=year)
    if imdb_meta:
        if 'language' not in imdb_meta or not imdb_meta['language']:
            cast_str = ', '.join(imdb_meta.get('cast', []))
            lang = infer_language_with_llm(imdb_meta.get('title', base_title), imdb_meta.get('year', year), cast_str)
            if lang:
                imdb_meta['language'] = lang
                imdb_meta['languages'] = [lang]
        return imdb_meta

    return {'error': 'No IMDb or Wikipedia metadata found', 'title': movie_name}

# Example usage
movie = 'Vadakkan (2025)'
metadata = get_movie_metadata(movie)
print(metadata)

In [ ]:
metadata = df['cleaned_name'].progress_apply(get_movie_metadata)

In [ ]:
def get_language(movie_name):
    metadata = get_movie_metadata(movie_name)
    if 'language' in metadata:
        return metadata['language']
    return metadata.get("languages", [""])[0] if "languages" in metadata else ''

df['language'] = df['cleaned_name'].progress_apply(get_language)
df.head()

In [ ]:
df.to_csv(f'renamed_files_{datetime.now()}.csv', index=False)

In [ ]:
df = pd.read_csv('renamed_files_2025-10-07 11:53:38.485638.csv')
df

In [ ]:
apply_renaming(df, base_path)

In [ ]:
create_folder_movies(base_path)

In [ ]:
create_folder_S01(base_path)

In [ ]:
metadata

In [ ]:
df[(df['language'] == 'English') & (df['content_type'] == 'web series')]

In [2]:
df = pd.read_csv('media_organization_plan_2026-11-18_15-58-33.csv')

In [3]:
df

,Source Path,Original Name,Content Type,Title,Year,Season,Episode,Episode Title,Primary Language,Target Directory,Target File Path,Status
0,/Volumes/Media_Temp/Torrent/Happy Ending 2025 ...,Happy Ending 2025 1080p Tagalog WEB-DL HEVC x2...,movie,Happy Ending,2025.0,NaN,NaN,NaN,English,/Volumes/Media_Temp/Clean/Movies/English/Happy...,/Volumes/Media_Temp/Clean/Movies/English/Happy...,PLANNED
1,/Volumes/Media_Temp/Torrent/Sundutan 2026 1080...,Sundutan 2026 1080p Tagalog WEB-DL HEVC x265 B...,movie,Sundutan,2026.0,NaN,NaN,NaN,Tagalog,/Volumes/Media_Temp/Clean/Movies/Tagalog/Sundu...,/Volumes/Media_Temp/Clean/Movies/Tagalog/Sundu...,PLANNED
2,/Volumes/Media_Temp/Torrent/Tayuan 2 2026 1080...,Tayuan 2 2026 1080p Tagalog WEB-DL HEVC x265 B...,movie,Tayuan 2,2026.0,NaN,NaN,NaN,Tagalog,/Volumes/Media_Temp/Clean/Movies/Tagalog/Tayua...,/Volumes/Media_Temp/Clean/Movies/Tagalog/Tayua...,PLANNED
3,/Volumes/Media_Temp/Torrent/Tayuan 2023 1080p ...,Tayuan 2023 1080p Tagalog WEB-DL HEVC x265 BON...,movie,Tayuan,2023.0,NaN,NaN,NaN,Tagalog,/Volumes/Media_Temp/Clean/Movies/Tagalog/Tayua...,/Volumes/Media_Temp/Clean/Movies/Tagalog/Tayua...,PLANNED
4,/Volumes/Media_Temp/Torrent/The.Drama.2026.PL....,The.Drama.2026.PL.1080p.AMZN.WEB-DL.XviD-MAXX.avi,movie,The Drama,2026.0,NaN,NaN,NaN,English,/Volumes/Media_Temp/Clean/Movies/English/The D...,/Volumes/Media_Temp/Clean/Movies/English/The D...,PLANNED
...,...,...,...,...,...,...,...,...,...,...,...,...
760,/Volumes/Media_Temp/Torrent/www.1TamilMV.lc - ...,www.1TamilMV.lc - Honeymoon Se Hatya S01E02.mkv,web series,Honeymoon Se Hatya,2025.0,1.0,2.0,NaN,Hindi,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,PLANNED
761,/Volumes/Media_Temp/Torrent/www.1TamilMV.lc - ...,www.1TamilMV.lc - Honeymoon Se Hatya S01E03.mkv,web series,Honeymoon Se Hatya,2025.0,1.0,3.0,NaN,Hindi,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,PLANNED
762,/Volumes/Media_Temp/Torrent/www.1TamilMV.lc - ...,www.1TamilMV.lc - Honeymoon Se Hatya S01E04.mkv,web series,Honeymoon Se Hatya,2025.0,1.0,4.0,NaN,Hindi,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,PLANNED
763,/Volumes/Media_Temp/Torrent/www.1TamilMV.lc - ...,www.1TamilMV.lc - Honeymoon Se Hatya S01E05.mkv,web series,Honeymoon Se Hatya,2025.0,1.0,5.0,NaN,Hindi,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,/Volumes/Media_Temp/Clean/Web Series/Hindi/Hon...,PLANNED
